In [2]:
from transformers import AutoModelForCausalLM, AutoTokenizer

model_name = 'Qwen/Qwen3-0.6B'

tokenizer = AutoTokenizer.from_pretrained(model_name)

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

In [3]:
from pathlib import Path

from peft import PeftModel
from transformers import BitsAndBytesConfig
import torch

bnb_config = BitsAndBytesConfig(
    load_in_4bit = True,
    bnb_4bit_quant_type = 'nf4',
    bnb_4bit_compute_dtype=torch.float32,
    bnb_4bit_use_double_quant=True,
)

q_model = AutoModelForCausalLM.from_pretrained(
    model_name, 
    quantization_config=bnb_config,
    device_map={"": 0},
    # attn_implementation=''
    trust_remote_code=True
    )

adapter_path = Path("Qwen3_0_6B_xlam/checkpoint-1000").resolve()
adapter_config_path = adapter_path / "adapter_config.json"
if not adapter_config_path.is_file():
    raise FileNotFoundError(
        f"LoRA adapter not found at {adapter_config_path}. "
        "Set adapter_path to the checkpoint produced by fine-tuning."
    )

model = PeftModel.from_pretrained(q_model, str(adapter_path))
model.eval()
print(f"Loaded fine-tuned LoRA adapter from {adapter_path}")

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

In [4]:
import torch

def get_model_size_mb(model):
    # Sum up parameters and buffers
    param_size = sum(p.nelement() * p.element_size() for p in model.parameters())
    buffer_size = sum(b.nelement() * b.element_size() for b in model.buffers())
    
    total_size_bytes = param_size + buffer_size
    total_size_mb = total_size_bytes / (1024 ** 2)
    return total_size_mb

print(f"Full model size: {get_model_size_mb(model)}")
print(f"4bit Quantized model size: {get_model_size_mb(q_model)}")

Full model size: 1136.87548828125
4bit Quantized model size: 506.87548828125


In [5]:
from datasets import Dataset, load_dataset
import json

def process_xlam_sample(row, tokenizer):
    formatted_query = f"<user>{row['query']}</user>\n\n"

    try:
        parsed_tools = json.loads(row["tools"])
    except json.JSONDecodeError as exc:
        raise ValueError("Invalid JSON in the source tools field") from exc

    try:
        parsed_answers = json.loads(row["answers"])
    except json.JSONDecodeError as exc:
        raise ValueError("Invalid JSON in the source answers field") from exc

    tools_text = json.dumps(
        parsed_tools,
        ensure_ascii=False,
        separators=(",", ":"),
    )
    answers_text = json.dumps(
        parsed_answers,
        ensure_ascii=False,
        separators=(",", ":"),
    )

    complete_text = (
        f"{formatted_query}"
        f"<tools>{tools_text}</tools>\n\n"
        f"<calls>{answers_text}</calls>"
        f"{tokenizer.eos_token}"
    )

    return {"text": complete_text}

In [14]:
from typing import Optional
import multiprocessing


def load_and_process_xlam_dataset(tokenizer: AutoTokenizer, sample_size: Optional[int] = None, split: str = None) -> Dataset:
    '''Load and process the complete xLAM dataset for function calling training'''
    
    print("Loading xLAM dataset ...")

    dataset = load_dataset('json', data_files='../data/xlam_function_calling_60k.json')
    split_dataset = dataset['train'].train_test_split(test_size=0.3, seed=42)
    train_dataset = split_dataset['train']
    test_dataset = split_dataset['test']

    # dataset = load_dataset("Salesforce/xlam-function-calling-60k", split='train')

    print(f"original dataset size: {len(dataset):,} samples")

    print("Processing into training format...")

    def process_batch(batch):
        processed_texts = []

        for i in range(len(batch["query"])):
            row = {
                "query": batch["query"][i],
                "tools": batch["tools"][i],
                "answers": batch["answers"][i],
            }

            processed_row = process_xlam_sample(row, tokenizer)
            processed_texts.append(processed_row["text"])

        return {"text": processed_texts}
    
    # Process the dataset
    if split:
        if split == 'train': 
            if sample_size and sample_size < len(train_dataset):
                train_dataset = train_dataset.select(range(sample_size))
                print(f"Using sample size: {sample_size} samples")   
            processed_dataset = train_dataset.map(
                process_batch,
                batched=True,
                batch_size=1000,
                num_proc=min(4, multiprocessing.cpu_count()),
                desc='Processing xLAM samples'
            )
        elif split == 'test' or split == 'val':
            if sample_size and sample_size < len(test_dataset):
                test_dataset = test_dataset.select(range(sample_size))
                print(f"Using sample size: {sample_size} samples")   
            processed_dataset = test_dataset.map(
                process_batch,
                batched=True,
                batch_size=1000,
                num_proc=min(4, multiprocessing.cpu_count()),
                desc='Processing xLAM samples'
            )
    else:
        if sample_size and sample_size < len(dataset):
            dataset = dataset.select(range(sample_size))
            print(f"Using sample size: {sample_size} samples")   
        processed_dataset = dataset.map(
            process_batch,
            batched=True,
            batch_size=1000,
            num_proc=min(4, multiprocessing.cpu_count()),
            desc='Processing xLAM samples'
        )

    print("Dataset processing complete")
    print(f"Final dataset size: {len(processed_dataset):,} samples")
    # print(f"Average text length: {sum(len(text) for text in processed_dataset['text']) / len(processed_dataset):,.0f} characters")

    return processed_dataset

processed_dataset = load_and_process_xlam_dataset(tokenizer=tokenizer, split='test', sample_size=100)

Loading xLAM dataset ...
original dataset size: 1 samples
Processing into training format...
Using sample size: 100 samples


Processing xLAM samples (num_proc=4):   0%|          | 0/100 [00:00<?, ? examples/s]

Dataset processing complete
Final dataset size: 100 samples


In [7]:
from tqdm.auto import tqdm


def run_batch_inference(
    model,
    tokenizer,
    prompts: list[str],
    batch_size: int = 8,
    max_new_tokens: int = 128,
) -> list[str]:
    if batch_size <= 0:
        raise ValueError("batch_size must be greater than zero")

    model.eval()
    tokenizer.padding_side = "left"

    if tokenizer.pad_token_id is None:
        tokenizer.pad_token = tokenizer.eos_token

    predictions = []

    total_batches = (len(prompts) + batch_size - 1) // batch_size
    progress = tqdm(
        range(0, len(prompts), batch_size),
        total=total_batches,
        desc="Batch inference",
        unit="batch",
    )

    for start in progress:
        prompt_batch = prompts[start : start + batch_size]

        inputs = tokenizer(
            prompt_batch,
            return_tensors="pt",
            padding=True,
            truncation=True,
        ).to(model.device)

        with torch.inference_mode():
            generated = model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=False,
                pad_token_id=tokenizer.pad_token_id,
            )

        generated_tokens = generated[:, inputs["input_ids"].shape[1] :]
        decoded_predictions = tokenizer.batch_decode(
            generated_tokens,
            skip_special_tokens=True,
        )
        batch_predictions = []
        for completion in decoded_predictions:
            call_content, closing_tag, _ = completion.partition("</calls>")
            if closing_tag:
                batch_predictions.append(
                    f"<calls>{call_content}</calls>"
                )
            else:
                batch_predictions.append(f"<calls>{call_content}")
        if len(batch_predictions) != len(prompt_batch):
            raise RuntimeError(
                "The model returned a different number of predictions "
                "than prompts in the batch"
            )

        predictions.extend(batch_predictions)
        progress.set_postfix(
            batch_size=len(prompt_batch),
            samples=f"{len(predictions)}/{len(prompts)}",
        )

    return predictions

from typing import Any
import json

def create_inference_prompt(
    query: str,
    tools: str | list[dict[str, Any]],
) -> str:
    """Format an xLAM prompt without exposing the ground-truth answer."""
    if isinstance(tools, str):
        try:
            parsed_tools = json.loads(tools)
        except json.JSONDecodeError as exc:
            raise ValueError("Invalid JSON in the tools field") from exc
    else:
        parsed_tools = tools

    tools_text = json.dumps(
        parsed_tools,
        ensure_ascii=False,
        separators=(",", ":"),
    )

    return (
        f"<user>{query}</user>\n\n"
        f"<tools>{tools_text}</tools>\n\n"
        "<calls>"
    )
    
# prompts = [
#     create_inference_prompt(
#         query=example["query"],
#         tools=example["tools"],
#     )
#     for example in processed_dataset.select(range(20))
# ]

# predictions = run_batch_inference(
#     model=model,
#     tokenizer=tokenizer,
#     prompts=prompts,
#     batch_size=8,
# )

### Evaluate Results and Calculate Accuracy

In [ ]:
import csv
import json
import re
from collections.abc import Mapping, Sequence
from pathlib import Path
from typing import Any


_CALLS_PATTERN = re.compile(r"<calls\b[^>]*>(.*?)</calls>", re.DOTALL | re.IGNORECASE)
_USER_PATTERN = re.compile(r"<user\b[^>]*>(.*?)</user>", re.DOTALL | re.IGNORECASE)


def _normalize_tool_call(call: Any, *, source: str) -> dict[str, Any]:
    """Normalize a tool call to {'name': str, 'arguments': dict}."""
    if not isinstance(call, Mapping):
        raise ValueError(f"{source} must be a JSON object, got {type(call).__name__}")

    # Also accept the OpenAI-style {'function': {'name': ..., 'arguments': ...}} shape.
    function = call.get("function")
    if isinstance(function, Mapping):
        call = function

    # Some generated examples use 'method'; normalize it to the required 'name'.
    name = call.get("name", call.get("method"))
    if not isinstance(name, str) or not name.strip():
        raise ValueError(f"{source} is missing a non-empty tool name: {call!r}")

    arguments = call.get("arguments", {})
    if isinstance(arguments, str):
        try:
            arguments = json.loads(arguments)
        except json.JSONDecodeError as exc:
            raise ValueError(f"{source} has invalid JSON arguments: {arguments!r}") from exc

    if not isinstance(arguments, Mapping):
        raise ValueError(f"{source} arguments must be a JSON object: {arguments!r}")

    return {
        "name": name.strip(),
        "arguments": dict(arguments),
    }


def _parse_tool_calls(value: Any, *, source: str, require_tags: bool = False) -> list[dict[str, Any]]:
    """Parse one or more JSON tool calls from raw JSON or <calls> blocks."""
    payloads: list[Any] = []

    if isinstance(value, str):
        call_blocks = _CALLS_PATTERN.findall(value)
        if require_tags and not call_blocks:
            raise ValueError(f"{source} contains no <calls>...</calls> block")

        serialized_payloads = call_blocks or [value]
        for block_index, serialized in enumerate(serialized_payloads):
            try:
                payloads.append(json.loads(serialized.strip()))
            except json.JSONDecodeError as exc:
                raise ValueError(
                    f"{source} call block {block_index} is not valid JSON: {serialized!r}"
                ) from exc
    else:
        payloads.append(value)

    normalized_calls: list[dict[str, Any]] = []
    for payload_index, payload in enumerate(payloads):
        calls = payload if isinstance(payload, list) else [payload]
        for call_index, call in enumerate(calls):
            normalized_calls.append(
                _normalize_tool_call(
                    call,
                    source=f"{source} payload {payload_index}, call {call_index}",
                )
            )

    return normalized_calls


def compare_tool_calls(
    predicted_tool_calls: Sequence[Mapping[str, Any]],
    actual_tool_calls: Sequence[Mapping[str, Any]],
    *,
    prediction_parse_error: str | None = None,
) -> dict[str, bool]:
    """Compare ordered tool names and arguments for one prediction."""
    if prediction_parse_error is not None:
        return {
            "tool_calls_match": False,
            "arguments_match": False,
            "exact_match": False,
        }

    predicted_names = [call["name"] for call in predicted_tool_calls]
    actual_names = [call["name"] for call in actual_tool_calls]
    predicted_arguments = [call["arguments"] for call in predicted_tool_calls]
    actual_arguments = [call["arguments"] for call in actual_tool_calls]

    tool_calls_match = predicted_names == actual_names
    arguments_match = predicted_arguments == actual_arguments
    return {
        "tool_calls_match": tool_calls_match,
        "arguments_match": arguments_match,
        "exact_match": tool_calls_match and arguments_match,
    }


def calculate_function_call_accuracy(
    evaluation_records: Sequence[Mapping[str, Any]],
) -> dict[str, int | float]:
    """Calculate tool-name, argument, and exact-match accuracy."""
    if not evaluation_records:
        raise ValueError("evaluation_records must contain at least one record")

    total = len(evaluation_records)
    tool_call_matches = sum(bool(record["tool_calls_match"]) for record in evaluation_records)
    argument_matches = sum(bool(record["arguments_match"]) for record in evaluation_records)
    exact_matches = sum(bool(record["exact_match"]) for record in evaluation_records)
    return {
        "total_examples": total,
        "tool_call_matches": tool_call_matches,
        "tool_call_accuracy": tool_call_matches / total,
        "argument_matches": argument_matches,
        "argument_accuracy": argument_matches / total,
        "exact_matches": exact_matches,
        "exact_match_accuracy": exact_matches / total,
    }


def evaluate_function_call_predictions(
    test_dataset: Dataset,
    prediction_results: Sequence[str],
) -> list[dict[str, Any]]:
    """Pair prompts and ground-truth calls with calls generated by the model.

    The dataset and prediction list must be positionally aligned. Ground truth
    is read from the `answers` column when present, otherwise from the dataset's
    formatted `text` column. Predictions are parsed from every <calls> block.
    """
    if isinstance(prediction_results, (str, bytes)):
        raise TypeError("prediction_results must be a sequence of response strings")

    if len(test_dataset) != len(prediction_results):
        raise ValueError(
            "test_dataset and prediction_results must have the same length: "
            f"{len(test_dataset)} != {len(prediction_results)}"
        )

    evaluation_records: list[dict[str, Any]] = []

    for index, (example, prediction) in enumerate(zip(test_dataset, prediction_results, strict=True)):
        text = example.get("text", "")

        prompt = example.get("query")
        if not isinstance(prompt, str) or not prompt.strip():
            user_matches = _USER_PATTERN.findall(text)
            if not user_matches:
                raise ValueError(f"test_dataset[{index}] has no query or <user> block")
            prompt = user_matches[0]

        answers = example.get("answers")
        ground_truth_from_text = answers is None
        if answers is None:
            answers = text

        prediction_parse_error = None
        try:
            predicted_tool_calls = _parse_tool_calls(
                prediction,
                source=f"prediction_results[{index}]",
                require_tags=True,
            )
        except ValueError as exc:
            predicted_tool_calls = []
            prediction_parse_error = str(exc)

        actual_tool_calls = _parse_tool_calls(
            answers,
            source=f"test_dataset[{index}] ground truth",
            require_tags=ground_truth_from_text,
        )
        comparison = compare_tool_calls(
            predicted_tool_calls,
            actual_tool_calls,
            prediction_parse_error=prediction_parse_error,
        )

        evaluation_records.append({
            "user_prompt": prompt.strip(),
            "predicted_tool_calls": predicted_tool_calls,
            "actual_tool_calls": actual_tool_calls,
            **comparison,
            "prediction_parse_error": prediction_parse_error,
            "predicted_raw": prediction if prediction_parse_error else None,
        })

    return evaluation_records


def save_accuracy_summary(
    accuracy_metrics: Mapping[str, int | float],
    output_path: str | Path = "tool_call_eval_accuracy_qwen-3-0.6B.json",
) -> Path:
    """Write aggregate function-call accuracy metrics to a JSON file."""
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    with output_path.open("w", encoding="utf-8") as summary_file:
        json.dump(dict(accuracy_metrics), summary_file, ensure_ascii=False, indent=2)
    print(f"Saved accuracy summary to {output_path.resolve()}")
    return output_path


def save_function_call_evaluation(
    evaluation_records: Sequence[Mapping[str, Any]],
    output_stem: str | Path = "tool_call_eval_results_qwen-3-0.6B",
) -> dict[str, Path]:
    """Write evaluation records to CSV and lossless JSON files."""
    output_stem = Path(output_stem)
    output_stem.parent.mkdir(parents=True, exist_ok=True)
    csv_path = output_stem.parent / f"{output_stem.name}.csv"
    json_path = output_stem.parent / f"{output_stem.name}.json"

    normalized_records = [dict(record) for record in evaluation_records]
    required_fields = {
        "user_prompt",
        "predicted_tool_calls",
        "actual_tool_calls",
        "tool_calls_match",
        "arguments_match",
        "exact_match",
    }
    for index, record in enumerate(normalized_records):
        missing_fields = required_fields - record.keys()
        if missing_fields:
            raise ValueError(
                f"evaluation_records[{index}] is missing fields: "
                f"{sorted(missing_fields)}"
            )

    with csv_path.open("w", encoding="utf-8", newline="") as csv_file:
        writer = csv.DictWriter(
            csv_file,
            fieldnames=[
                "user_prompt",
                "predicted_tool_calls",
                "actual_tool_calls",
                "tool_calls_match",
                "arguments_match",
                "exact_match",
                "prediction_parse_error",
                "predicted_raw",
            ],
        )
        writer.writeheader()
        for record in normalized_records:
            writer.writerow({
                "user_prompt": record["user_prompt"].replace(
                    "\r", "\\r"
                ).replace("\n", "\\n"),
                "predicted_tool_calls": json.dumps(
                    record["predicted_tool_calls"], ensure_ascii=False
                ),
                "actual_tool_calls": json.dumps(
                    record["actual_tool_calls"], ensure_ascii=False
                ),
                "tool_calls_match": record["tool_calls_match"],
                "arguments_match": record["arguments_match"],
                "exact_match": record["exact_match"],
                "prediction_parse_error": (
                    record.get("prediction_parse_error") or ""
                ).replace("\r", "\\r").replace("\n", "\\n"),
                "predicted_raw": json.dumps(
                    record["predicted_raw"], ensure_ascii=False
                ) if record.get("predicted_raw") is not None else "",
            })

    with json_path.open("w", encoding="utf-8") as json_file:
        json.dump(normalized_records, json_file, ensure_ascii=False, indent=2)

    print(f"Saved {len(normalized_records):,} records to {csv_path.resolve()}")
    print(f"Saved {len(normalized_records):,} records to {json_path.resolve()}")
    return {"csv": csv_path, "json": json_path}

def process_test_dataset(dataset_, sample_size=None):
    output = []
    if sample_size and sample_size < len(dataset_):
        sample_size = sample_size
    else:
        sample_size = len(dataset_)

    for idx in range(sample_size):
        sample = dataset_[idx]
        output.append({
            'id': sample['id'],
            'query': sample['query'],
            'answers': sample['answers'],
            'tools': sample['tools'],
            'text': sample['text']
        })

    return output


prompts = [
    create_inference_prompt(
        query=example["query"],
        tools=example["tools"],
    )
    for example in processed_dataset
]

results = run_batch_inference(
    model=model,
    tokenizer=tokenizer,
    prompts=prompts,
    batch_size=8,
)


# test_dataset = process_test_dataset(processed_dataset)

# evaluation_results = evaluate_function_call_predictions(test_dataset, results)

# Use the same full dataset passed to test_function_calling_examples.
evaluation_results = evaluate_function_call_predictions(test_dataset, results)
accuracy_metrics = calculate_function_call_accuracy(evaluation_results)
print(json.dumps(accuracy_metrics, indent=2))
accuracy_summary_path = save_accuracy_summary(accuracy_metrics)
evaluation_files = save_function_call_evaluation(evaluation_results)
evaluation_results[:2]


Batch inference:   0%|          | 0/13 [00:00<?, ?batch/s]

TypeError: process_test_dataset() missing 1 required positional argument: 'sample_size'